In [2]:
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.training.errors import ErrorDict
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.data.hamiltonian_dataset import seeded_random_split
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    dftpy_grid, CubicalGrid, spherical_grid, spherical_radial_sampling
from equiv_dens.training.model_loader import load_model
import equiv_dens.utils.base as utils
from equiv_dens.utils import orbitals
from functools import partial
import os
import numpy as np

%load_ext autoreload
%autoreload 2

Use "numpy" for Fourier Transform


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


In [22]:
args, hyperparam_args = parse_command_line_arguments(arg_file='H_dens.txt')

print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = False
# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")

args.verbose = 0
args.use_gpu = False
args.cube_grid = True
args.radii_adjust = True 
if args.cube_grid:
    args.cube_origin = -3
    args.cube_extent = 6
    args.cube_size = 50
    args.radii_adjust = False
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

    
dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=torch.float32,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=0,
                           radii_adjust=args.radii_adjust)

dataset_df = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=torch.float32,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=0,
                           radii_adjust=args.radii_adjust,
                           projected_density=True)
    
dataset_df.density_fitting[0]['df_coeff'][11:] = 0
sample = dataset.get_properties(0)
sample_df = dataset_df.get_properties(0)
print('density loss', torch.sum(torch.abs(sample['density'] - sample_df['density']) * sample['coord_weights'])/sample['atom_numbers'])
print('density integral', torch.sum(sample['density'] * sample['coord_weights']))
print('df integral', torch.sum(sample_df['density'] * sample['coord_weights']))
print(dataset_df.density_fitting)
print(dataset.atoms)
model = load_model(args, dataset)
result = model(sample)
print('ml density loss', torch.sum(torch.abs(sample['density'] - result['density']) * sample['coord_weights'])/sample['atom_numbers'])
print('ml density integral', torch.sum(result['density'] * sample['coord_weights']))


type dtype <class 'torch.dtype'>
args np dir datasets/H_augccpvdz.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperparam arg in

In [11]:
atom_types = ['H', 'C', 'N', 'O', 'S']
atom_numbers = [1, 6, 7, 8, 16]
L0_coeffs = {'spherical_coeffs': {}, 'radial_width': {}, 'radial_scale': {}}

for i in range(len(atom_types)):
    at = atom_types[i]
    an = atom_numbers[i]
    args, hyperparam_args = parse_command_line_arguments(arg_file=(at + '_dens.txt'))
    print('type dtype', type(args.dtype))
    args.fix_arguments = True
    print('args np dir', args.np_dataset)
    # no restart directory specified
    directory = args.restart  # load directory name
    # load latest checkpoint
    checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
    checkpoint = torch.load(os.path.join(
        checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
    latest_checkpoint = checkpoint['step']
    model_code = checkpoint['ID']  # load ID
    step = checkpoint['step']
    for arg in vars(checkpoint['args']):
        if args.fix_arguments:
            if arg in hyperparam_args:
                print('loading hyperparam arg', arg)
                setattr(args, arg, getattr(checkpoint['args'], arg))
        else:
            print('loading all arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    restore = True

    args.best_model_path = 'best_' + model_code + '.pth'
    print('best_model_path', args.best_model_path)

    print('model code:', model_code)
    # determine whether GPU is used for training
    print('args use gpu', args.use_gpu)
    args.use_gpu = False
    # load dataset(s)
    print("loading density from" + str(args.dens_dataset) + "...")
    print("loading atoms from" + args.np_dataset + "...")

    args.verbose = 0
    args.use_gpu = False
    args.radii_adjust = False 
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None


    dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                               orbitals_path=args.orbitals_file,
                               density_n_samp=10000000000,
                               required_properties=['density'],
                               center_positions=False,
                               radial_coeffs_file=args.radial_coeffs_file,
                               dtype=torch.float32,
                               grid_fn=grid_fn,
                               sampling_fn=sampling_fn,
                               grid_extent=grid_extent,
                               grid_origin=grid_origin,
                               verbose=0,
                               radii_adjust=args.radii_adjust)
    
    model = load_model(args, dataset)
    sample = dataset.get_properties([0])
    results = model(sample)
    print('sph coeffs', results['spherical_coeffs'])
    print('scale coeffs', results['radial_scale'])
    print('width coeffs', results['radial_width'])
    L0_coeffs['spherical_coeffs'][at] = results['spherical_coeffs'][0][(an, 0)].squeeze(0).squeeze(0)
    L0_coeffs['radial_width'][at] = results['radial_width'][0][(an, 0)].squeeze(0).squeeze(0)
    L0_coeffs['radial_scale'][at] = results['radial_scale'][0][(an, 0)].squeeze(0).squeeze(0)
    
    print(L0_coeffs['spherical_coeffs'][at].shape)
    print('L0_coeffs', L0_coeffs)
    
np.save('datasets/augccpvqzjkfit_init_L0_free.npy', L0_coeffs, allow_pickle=True)

type dtype <class 'torch.dtype'>
args np dir datasets/H_augccpvdz.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperparam arg in